In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Define data paths
RAW_DATA_PATH = Path('../data/raw')
PROCESSED_DATA_PATH = Path('../data/processed')

# Create processed data directory if it doesn't exist
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

def clean_weather_data(df):
    """Clean and preprocess weather data."""
    # Drop duplicates
    df = df.drop_duplicates()
    
    # Convert timestamp to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Handle missing values
    df = df.fillna({
        'temperature': df['temperature'].mean(),
        'humidity': df['humidity'].mean(),
        'pressure': df['pressure'].mean(),
        'wind_speed': df['wind_speed'].mean()
    })
    
    # Remove outliers using IQR method
    for column in ['temperature', 'humidity', 'pressure', 'wind_speed']:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        df = df[~((df[column] < (Q1 - 1.5 * IQR)) | (df[column] > (Q3 + 1.5 * IQR)))]
    
    return df

# Load raw data
raw_files = list(RAW_DATA_PATH.glob('*.csv'))
print(f"Found {len(raw_files)} raw data files")

# Process each file
for file_path in raw_files:
    print(f"Processing {file_path.name}")
    
    # Read raw data
    df = pd.read_csv(file_path)
    
    # Clean and process data
    processed_df = clean_weather_data(df)
    
    # Save processed data
    output_file = PROCESSED_DATA_PATH / f"processed_{file_path.name}"
    processed_df.to_csv(output_file, index=False)
    
    # Also save as parquet for efficient storage and reading
    parquet_file = output_file.with_suffix('.parquet')
    processed_df.to_parquet(parquet_file, index=False)
    
    print(f"Saved processed data to {output_file} and {parquet_file}")

print("Data processing completed!")
